In [ ]:
# Cellule masquée à l'export (--no-input). Elle charge les résultats déjà produits
# par probleme1.py et probleme2.py et définit la mise en forme. Aucun nouveau calcul.
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display

R1 = Path("resultats_probleme1")   # résultats du Problème 1
R2 = Path("resultats_probleme2")   # résultats du Problème 2


def fr(x, d=2):
    """Nombre au format français (virgule, espace insécable pour les milliers, signe −).
    Arrondi décimal exact (moitié vers le haut), pour que 4,1350 donne 4,14."""
    if pd.isna(x):
        return ""
    q = Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP)
    return f"{q:,.{d}f}".replace(",", " ").replace(".", ",").replace("-", "−")


def fr_p(p):
    """Valeur p au format français, « < 0,0001 » si elle est très petite."""
    return "&lt; 0,0001" if p < 1e-4 else fr(p, 4)


def tableau(df):
    """Affiche un tableau déjà mis en forme."""
    df = df.copy()
    df.index.name = None
    display(HTML(df.to_html(escape=False, border=0, classes="tab")))


def figure(fichier, largeur=680):
    """Affiche un graphique PNG produit par le code."""
    display(Image(filename=str(fichier), width=largeur))


def apercu(donnees, stats):
    """Cinq premiers mois (une ligne par série) et statistiques descriptives."""
    premiers = donnees.head().T
    st = stats[["mean", "std", "min", "max", "asymetrie", "kurtosis_exces"]].copy()
    st.columns = ["Moyenne", "Éc.-type", "Min.", "Max.", "Asym.", "Kurt."]
    t = pd.concat({"Cinq premiers mois": premiers, "Statistiques, 504 mois": st}, axis=1)
    return t.map(fr)


display(HTML("""
<style>
table.tab {border-collapse: collapse; font-size: 9pt; margin: 4px auto;}
table.tab th, table.tab td {padding: 2px 7px; text-align: right; border: none;}
table.tab thead tr:first-child th {border-top: 1.5px solid black;}
table.tab thead tr:last-child th {border-bottom: 1px solid black;}
table.tab tbody tr:last-child td, table.tab tbody tr:last-child th {border-bottom: 1.5px solid black;}
table.tab tbody th {text-align: left; font-weight: normal;}
</style>
"""))

fac = pd.read_csv(R1 / "Q1a_factors_exces_rendements_mensuels_1980-2021.csv", index_col=0)
port = pd.read_csv(R1 / "Q1a_portfolios_exces_rendements_mensuels_1980-2021.csv", index_col=0)
stats_fac = pd.read_csv(R1 / "Q1a_factors_statistiques_descriptives.csv", index_col=0)
stats_port = pd.read_csv(R1 / "Q1a_portfolios_statistiques_descriptives.csv", index_col=0)
lam = pd.read_csv(R1 / "Q1c_primes_de_risque_lambda.csv", header=[0, 1], index_col=0)
betas = pd.read_csv(R1 / "Q1c_betas_expositions.csv", index_col=0)
betas_t = pd.read_csv(R1 / "Q1c_betas_t_stats_robustes.csv", index_col=0)
r2_j = pd.read_csv(R1 / "Q1c_R2_et_tests_J.csv", index_col=0)
llf_q2a = pd.read_csv(R2 / "Q2a_log_vraisemblances_dof5_dof30.csv", index_col=0)
rv_q2b = pd.read_csv(R2 / "Q2b_test_rapport_vraisemblance.csv", index_col=0)

# Devoir 4 – Méthodes économétriques en finance

MATH 60231, HEC Montréal

Professeur Anthony Sanford

Ulric Duranleau (11337928), Alicia Ponce (11318956) et Naomy Roy (11336597)

Novembre 2026

# Problème 1 – Tests de modèles d'évaluation d'actifs

## Problème 1a

La fonction `Q1_KenFrenchData` lit trois fichiers de la bibliothèque de Kenneth R. French (French, s. d.), soit les cinq facteurs de Fama et French (version 2x3), le facteur momentum et les 17 portefeuilles sectoriels. Elle retient les rendements mensuels de janvier 1980 à décembre 2021 (504 mois), en pourcentage par mois. Nous prenons les portefeuilles pondérés par la valeur, car ils reflètent le poids économique des entreprises.

Les six facteurs sont déjà des excédents de rendement. Mkt-RF est net du taux sans risque, et SMB, HML, RMW, CMA et Mom sont des portefeuilles longs-courts autofinancés dans lesquels ce taux s'annule. Nous soustrayons en revanche le taux RF du bon du Trésor à un mois des rendements bruts des 17 portefeuilles. Les tableaux 1 et 2 présentent les cinq premières lignes et les statistiques descriptives de `factors` (504 × 6) et de `portfolios` (504 × 17).

Tableau 1 – *Facteurs, cinq premiers mois et statistiques descriptives*

In [ ]:
tableau(apercu(fac, stats_fac))

*Note.* Excédents de rendement en % par mois, une ligne par facteur. Les statistiques portent sur les 504 mois de 1980-01 à 2021-12. Éc.-type, Asym. et Kurt. désignent l'écart-type, l'asymétrie et la kurtosis en excès (nulle pour la loi normale). Calculs des auteurs à partir de French (s. d.).

Les cinq premiers mois couvrent le début de la récession de 1980, de février à juillet selon le NBER (NBER, 2023). En mars 1980, le marché perd 12,89 % et Mom perd 9,59 % après des gains de 7,46 % et 7,91 % en janvier et février. Ce renversement illustre le risque d'une stratégie qui achète les gagnants récents quand le marché se retourne.

Sur 1980-2021, les six facteurs ont une moyenne positive, de 0,09 % par mois pour SMB à 0,74 % pour le marché. Tous ont une kurtosis en excès positive, donc des queues plus épaisses que la loi normale, un fait stylisé présenté à la séance 1 (van Norden et Sanford, 2023). Mom combine une forte kurtosis (10,20), la plus forte asymétrie négative (−1,31) et le pire mois (−34,36 %), ce qui révèle un risque de krach.

Tableau 2 – *Portefeuilles sectoriels, cinq premiers mois et statistiques descriptives*

In [ ]:
tableau(apercu(port, stats_port))

*Note.* Excédents de rendement (rendement brut moins RF) en % par mois, une ligne par portefeuille pondéré par la valeur. Les codes des industries sont ceux de French (s. d.). Les statistiques portent sur les 504 mois de 1980-01 à 2021-12. Éc.-type, Asym. et Kurt. désignent l'écart-type, l'asymétrie et la kurtosis en excès. Calculs des auteurs à partir de French (s. d.).

Tous les portefeuilles perdent en mars 1980, de −4,01 % (Cnsum) à −22,35 % (Mines), ce qui montre le poids du facteur marché commun. Oil gagne pourtant 12,17 % en février alors que le marché recule de 1,23 %, un signe que chaque secteur a aussi des expositions propres. Sur 1980-2021, les excédents moyens vont de 0,50 % par mois (Steel) à 0,95 % (Cars) et les écarts-types de 3,93 % (Utils) à 8,18 % (Steel). Les 17 portefeuilles ont une kurtosis en excès positive et 15 ont une asymétrie négative, donc des pertes extrêmes plus fréquentes que sous la loi normale.

## Problème 1b

La figure 1 présente les séries des six facteurs, la figure 2 leurs corrélations et la figure 3 les séries des 17 portefeuilles, chaque fois sur une même échelle verticale. Nous annualisons les écarts-types (× √12) et les moyennes (× 12) pour les exprimer en unités usuelles.

Mom (15,54 % par an) et Mkt-RF (15,51 %) sont les facteurs les plus volatils (figure 1), devant HML (10,49 %), SMB (10,04 %), RMW (8,32 %) et CMA (6,83 %). Les autres facteurs longs-courts sont moins volatils que le marché, car le risque de marché de leurs deux jambes se compense en partie. Le pire mois de Mom, avril 2009 (−34,36 %), survient à la fin de la Grande Récession (décembre 2007 à juin 2009), un épisode où les anciens titres perdants rebondissent brusquement. HML subit sa pire perte en mars 2020 (−13,83 %), pendant la récession de la COVID-19, tandis que le pire mois du marché, octobre 1987 (−23,19 %), ne coïncide avec aucune récession.

Les excédents moyens les plus élevés sont ceux de Cars (11,39 % par an), Rtail (11,35 %) et Cnstr (11,05 %), et les plus bas ceux de Steel (5,94 %), Durbl (7,27 %) et Mines (7,46 %) (figure 3). Ces écarts sont imprécis, car l'intervalle de confiance à 95 % de Steel (−2,65 % à 14,54 %) recoupe largement celui de Cars (3,89 % à 18,89 %). Steel et Mines sont aussi les plus volatils (28,35 % et 27,60 % par an), donc le risque total n'est pas rémunéré, ce qui motive les modèles factoriels du problème 1c. Neuf portefeuilles subissent leur pire mois en octobre 1987, hors récession, et quatre en octobre 2008, pendant la Grande Récession. Oil (−34,78 %) et Utils (−13,17 %) le subissent en mars 2020, pendant la récession de la COVID-19.

Figure 1 – *Rendements mensuels des six facteurs, 1980-2021*

In [ ]:
figure(R1 / "Q1b_fig1_series_temporelles_facteurs.png", 580)

*Note.* Excédents de rendement en % par mois. Les bandes grises sont les récessions américaines datées par le NBER (NBER, 2023). Chaque panneau indique la volatilité annualisée et le pire mois. Calculs des auteurs à partir de French (s. d.).

Figure 2 – *Corrélations entre les six facteurs, 1980-2021*

In [ ]:
figure(R1 / "Q1b_fig3_correlations_facteurs_heatmap.png", 380)

*Note.* Coefficients de corrélation de Pearson calculés sur les 504 rendements mensuels. Calculs des auteurs à partir de French (s. d.).

La structure de corrélation de la figure 2 suggère une colinéarité limitée. La seule corrélation élevée relie HML et CMA (0,66), ce qui suggère que les entreprises de valeur investissent peu. Suivent RMW et SMB (−0,42) puis CMA et Mkt-RF (−0,37), et toutes les autres sont inférieures à 0,26 en valeur absolue, car les facteurs sont construits pour capter des risques distincts. [À COMPLÉTER : facteurs d'inflation de la variance (VIF) des six facteurs, qui ne sont pas produits par le code actuel.] La colinéarité ne devrait donc pas empêcher l'estimation au problème 1c, mais elle rendra les bêtas sur HML et CMA moins précis.

Figure 3 – *Excédents de rendement mensuels des 17 portefeuilles sectoriels, 1980-2021*

In [ ]:
figure(R1 / "Q1b_fig2_series_temporelles_portefeuilles.png", 620)

*Note.* Excédents de rendement en % par mois, avec la même échelle verticale pour tous les panneaux. Les bandes grises sont les récessions du NBER (NBER, 2023). Calculs des auteurs à partir de French (s. d.).

## Problème 1c

La fonction `Q1_AssetPricingModel` estime le modèle en deux passes avec le package linearmodels (Sheppard, s. d.), présenté à la séance 8 (van Norden et Sanford, 2023). Elle affiche un résumé et retourne les primes de risque, les bêtas, leurs statistiques t, le R² transversal et le test J.

**(i)** La passe en séries temporelles régresse par MCO l'excédent de rendement de chaque portefeuille sur une constante et les six facteurs, ce qui donne les $\hat\beta_i$ et les $\hat\alpha_i$.

**(ii)** La passe transversale relie les rendements moyens $\bar r_i$ aux $\hat\beta_i$. Avec `Traded=False` (LinearFactorModel), nous régressons les 17 moyennes sur les $\hat\beta_i$ sans constante, puisque les actifs sont des excédents de rendement. Avec `Traded=True` (TradedFactorModel), chaque facteur est un portefeuille négocié et sa prime $\hat\lambda$ est sa moyenne temporelle.

Les statistiques t sont robustes à l'hétéroscédasticité (White, avec correction des degrés de liberté), car la variance des rendements varie dans le temps. Avec `Traded=False`, elles tiennent aussi compte du fait que les $\hat\beta_i$ sont estimés. Nous calculons nous-mêmes le R² transversal, $1-\sum_i \hat\alpha_i^2/\sum_i(\bar r_i-\bar{\bar r})^2$, car celui de linearmodels porte sur les régressions temporelles. Le test J porte sur l'hypothèse que toutes les erreurs de valorisation sont nulles. Il suit une loi χ²(N) = χ²(17) avec `Traded=True` et χ²(N − K) = χ²(11) avec `Traded=False`.

Tableau 3 – *Primes de risque estimées des six facteurs*

In [ ]:
t = {}
for v in ["Traded=True", "Traded=False"]:
    t[(v, "λ (%/mois)")] = lam[(v, "lambda_mens_%")].map(fr)
    t[(v, "λ (%/an)")] = lam[(v, "lambda_ann_%")].map(fr)
    t[(v, "t")] = lam[(v, "t_stat")].map(fr)
    t[(v, "p")] = lam[(v, "p_valeur")].map(fr_p)
tableau(pd.DataFrame(t))

*Note.* Primes en % par mois et annualisées (× 12). Statistiques t robustes à l'hétéroscédasticité et valeurs p bilatérales. `Traded=True` utilise TradedFactorModel et `Traded=False` utilise LinearFactorModel sans estimation du taux sans risque (Sheppard, s. d.). 504 mois et 17 portefeuilles.

Avec `Traded=True`, les primes du marché (0,74 % par mois, soit 8,92 % par an), de RMW, de CMA et de Mom sont significatives à 5 %. Avec `Traded=False`, seule celle du marché le reste (t = 3,99), et les primes de SMB, HML et CMA deviennent négatives alors que les moyennes de ces facteurs sont positives. Les 17 industries offrent trop peu de dispersion dans les bêtas autres que celui du marché pour estimer six primes avec précision.

Tableau 4 – *Expositions des portefeuilles aux facteurs (β) et statistiques t robustes*

In [ ]:
tableau(betas.map(fr) + " (" + betas_t.map(fr) + ")")

*Note.* Statistique t robuste à l'hétéroscédasticité entre parenthèses. Une valeur absolue supérieure à 1,96 indique un bêta significatif à 5 %. Ces estimations sont identiques avec `Traded=True` et `Traded=False`, car la passe en séries temporelles ne dépend pas de `Traded`. 504 mois.

Les bêtas de marché vont de 0,59 (Utils) à 1,33 (Steel), avec des t entre 13,19 et 69,39. Les autres expositions ont un sens économique. Finan se comporte comme un secteur de valeur (0,73 sur HML). Machn, qui inclut l'équipement informatique, a le profil d'un secteur de croissance (−0,36 sur HML et −0,37 sur RMW). Cnsum, qui inclut la pharmaceutique et le tabac, penche vers les grandes entreprises conservatrices (−0,30 sur SMB et 0,55 sur CMA).

Tableau 5 – *R² transversal et test J*

In [ ]:
lignes = {"R2_transversal": ("R² transversal", fr),
          "statistique_J": ("Statistique J", fr),
          "ddl_J": ("Degrés de liberté du J", lambda x: fr(x, 0)),
          "p_valeur_J": ("Valeur p du J", fr_p),
          "alpha_absolu_moyen_ann_%": ("Alpha absolu moyen (% par an)", fr)}
tableau(pd.DataFrame({lab: r2_j.loc[k].map(f) for k, (lab, f) in lignes.items()}).T)

*Note.* Sous l'hypothèse nulle que toutes les erreurs de valorisation sont nulles, J suit une loi χ²(17) avec `Traded=True` et χ²(11) avec `Traded=False`. L'alpha absolu moyen est la moyenne des |α̂ᵢ| des 17 portefeuilles, annualisée.

Avec `Traded=True`, le R² transversal est négatif (−2,36), car les primes imposées par les moyennes des facteurs prédisent des rendements moyens plus éloignés des rendements réalisés que leur simple moyenne commune. Le modèle surestime 12 des 17 rendements moyens et son erreur absolue moyenne atteint 2,38 % par an, contre 0,73 % avec `Traded=False`, dont le R² est de 0,69. Le test J rejette le modèle avec `Traded=True` (J = 40,81, p = 0,0010) mais pas avec `Traded=False` (J = 4,14, p = 0,9658). La figure 4 présente les erreurs de valorisation des deux versions.

Figure 4 – *Erreurs de valorisation (alphas) des 17 portefeuilles sectoriels*

In [ ]:
figure(R1 / "Q1c_fig4_alphas_IC95.png", 620)

*Note.* Alphas annualisés en % par an, avec des intervalles de confiance à 95 % fondés sur les t robustes. Les points orange sont significatifs à 5 %. Avec `Traded=True`, les alphas sont les interceptes des régressions temporelles. Avec `Traded=False`, ce sont les erreurs de la régression transversale.

## Problème 1d

**(i)** Avec `Traded=True`, les primes du marché (t = 3,73), de RMW (t = 3,52), de CMA (t = 2,86) et de Mom (t = 2,65) sont significatives à 5 %. Les investisseurs ont donc été rémunérés pour porter les risques de marché, de rentabilité, d'investissement et de momentum sur 1980-2021. SMB (t = 0,74) et HML (t = 1,50) n'ont pas de prime significative, ce qui concorde avec leurs faibles moyennes au tableau 1.

**(ii)** Le test J rejette l'hypothèse que tous les alphas sont nuls (J = 40,81 avec 17 degrés de liberté, p = 0,0010), donc les six facteurs n'expliquent pas toutes les différences de rendements moyens entre industries. Cinq portefeuilles ont un alpha significatif à 5 % (figure 4). Machn (+3,81 % par an) et Other (+2,02 %) rapportent plus que ne le prédit le modèle, tandis que Durbl (−4,23 %), Trans (−2,96 %) et Steel (−5,32 %) rapportent moins, ces deux derniers étant à la limite du seuil (t de −2,00 et −1,97).

**(iii)** Avec `Traded=True`, les primes ne sont pas estimées dans la coupe transversale. Chaque facteur est un portefeuille qui doit se valoriser lui-même avec un alpha nul, donc les primes sont fixées par les rendements moyens des facteurs et les 17 erreurs de valorisation restent libres, d'où la loi χ²(17). Avec `Traded=False`, les six primes sont choisies pour ajuster au mieux les 17 rendements moyens, ce qui rend les erreurs orthogonales aux six colonnes de bêtas, absorbe K = 6 degrés de liberté et laisse une loi χ²(11).

**(iv)** La version `Traded=True` est théoriquement appropriée, car les six facteurs sont des excédents de rendement de portefeuilles négociés et autofinancés, ce qui correspond au premier cas présenté à la séance 8. La théorie impose alors que la prime de chaque facteur égale son rendement moyen, une contrainte que `Traded=False` ignore. Les primes négatives de SMB, HML et CMA obtenues avec `Traded=False` contredisent d'ailleurs les moyennes positives de ces facteurs, et son J non significatif reflète un test moins exigeant plutôt qu'un meilleur modèle.

# Problème 2 – Estimation par maximum de vraisemblance

Nous régressons Mkt-RF sur une constante et SMB, HML, RMW, CMA et Mom, avec les 504 mois du problème 1 en % par mois. La fonction `Q2_OLSStudentT` calcule la log-vraisemblance

$$\ell(\text{dof}) = \sum_{t=1}^{T} \log f_{\text{dof}}\!\left(\frac{\hat e_t}{\hat\sigma}\right) - T\log\hat\sigma, \qquad \hat\sigma = \sqrt{SCR/T},$$

où $f_{\text{dof}}$ est la densité de Student et $\hat e_t$ les résidus MCO. Nous estimons $\hat\sigma$ par maximum de vraisemblance, l'approche d'estimation présentée à la séance 2 (van Norden et Sanford, 2023), ce qui traite aussi les deux modèles du test RV de la même façon. Quand dof tend vers l'infini, notre log-vraisemblance tend vers la log-vraisemblance gaussienne usuelle des MCO, qui utilise aussi SCR/T. Comme le prévoit l'énoncé, nous gardons la loi t(dof) standard sans la ramener à une variance de 1, ce qui serait d'ailleurs impossible pour dof ≤ 2, où la variance n'existe pas. Le terme −T log σ̂ vient du changement de variable entre $\hat e_t$ et $\hat e_t/\hat\sigma$ et rend comparables des modèles dont le σ̂ diffère.

## Problème 2a

Tableau 6 – *Log-vraisemblance du modèle avec dof = 5 et dof = 30*

In [ ]:
t = llf_q2a.rename(columns={"log_vraisemblance": "Log-vraisemblance"}).map(fr)
t.index = [f"dof = {d}" for d in llf_q2a.index]
tableau(t)

*Note.* Régression de Mkt-RF sur une constante et les cinq autres facteurs, avec des résidus standardisés supposés i.i.d. t(dof) et σ̂ = √(SCR/T). Rendements en % par mois, 504 mois.

La log-vraisemblance vaut −1 406,51 avec dof = 5 et −1 400,16 avec dof = 30. La loi t(30) décrit donc mieux les résidus, avec un gain de 6,35 points. Les résidus ont des queues plus épaisses que la loi normale, comme les rendements financiers en général (séance 1), mais moins épaisses que celles d'une t(5), qui place trop de probabilité dans les valeurs extrêmes. L'écart reflète aussi l'échelle, car une t(5) standard a une variance de 5/3 alors que les résidus standardisés ont une variance de 1.

## Problème 2b

Tableau 7 – *Test du rapport de vraisemblance de la nullité des cinq coefficients*

In [ ]:
t = pd.DataFrame({
    "llf non contraint": rv_q2b["llf_non_contraint"].map(fr),
    "llf contraint": rv_q2b["llf_contraint_constante_seule"].map(fr),
    "Statistique RV": rv_q2b["statistique_RV"].map(fr),
    "ddl": rv_q2b["ddl"].map(lambda x: fr(x, 0)),
    "Valeur p": rv_q2b["p_valeur"].astype(float).map(fr_p),
})
t.index = [f"dof = {d}" for d in rv_q2b.index]
tableau(t)

*Note.* Le modèle contraint régresse Mkt-RF sur une constante seulement. RV = 2(llf non contraint − llf contraint) suit une loi χ²(5) sous l'hypothèse nulle. σ̂ = √(SCR/T) dans les deux modèles.

**(i)** Sous l'hypothèse nulle, les cinq coefficients des facteurs sont nuls et le modèle contraint régresse Mkt-RF sur une constante seulement. Avec dof = 5, sa log-vraisemblance vaut −1 471,73, soit 65,21 points de moins que celle du modèle non contraint (−1 406,51). Les cinq facteurs expliquent donc une part importante des variations mensuelles du marché.

**(ii)** La statistique RV vaut 130,43 avec 5 degrés de liberté et sa valeur p est inférieure à 0,0001. Nous rejetons l'hypothèse nulle et concluons que les cinq facteurs expliquent conjointement l'excédent de rendement du marché. Ils sont toutefois mesurés le même mois que Mkt-RF, donc ils l'expliquent de façon contemporaine sans le prévoir.

**(iii)** Avec dof = 30, la statistique RV vaut 128,86, soit 1,57 de moins qu'avec dof = 5. Elle change parce que la log-vraisemblance dépend de la forme de la densité, évaluée aux résidus standardisés de chaque modèle. Sous la loi normale avec σ̂ du maximum de vraisemblance, elle vaudrait T log(SCR contraint / SCR non contraint) et ne dépendrait que du rapport des σ̂. La variation reste faible et la conclusion ne change pas.

## Problème 2c

Figure 5 – *Vraisemblance profilée des degrés de liberté*

In [ ]:
figure(R2 / "Q2c_fig1_vraisemblance_profilee.png", 500)

*Note.* Log-vraisemblance de la régression de Mkt-RF sur une constante et les cinq autres facteurs, évaluée sur la grille dof = e<sup>j/12</sup>, j = 0, …, 50 (de 1 à environ 64,5). Abscisse en échelle logarithmique. Le point orange marque le maximum de la grille. 504 mois, de 1980-01 à 2021-12.

**(i)** La courbe est unimodale. Elle monte fortement de dof = 1 (−1 511,27) jusqu'au sommet, puis redescend lentement jusqu'à dof ≈ 64,5 (−1 402,33). Elle est donc très asymétrique, avec une pente raide à gauche et un plateau à droite.

**(ii)** Le maximum est atteint à dof ≈ 14,4 (j = 32), avec une log-vraisemblance de −1 398,74. L'estimateur du maximum de vraisemblance des degrés de liberté vaut donc environ 14. Le sommet est plat, car la courbe reste à moins de 2 points du maximum entre dof ≈ 8,0 et dof ≈ 36,0. Les données situent donc dof dans cette zone sans le fixer avec précision.

## Problème 2d

**(i)** L'EMV est ≈ 15. Nous le lisons à l'abscisse du sommet de la courbe, qui se situe à dof ≈ 14,4, à l'intérieur de la grille. Les options ≈ 1 et > 60 correspondent aux deux extrémités de la grille, où la courbe est plus basse. L'option ≈ −1 370 est de l'ordre de grandeur d'une log-vraisemblance (nos valeurs sont proches de −1 400), pas d'un nombre de degrés de liberté, qui est forcément positif.

**(ii)** Un dof d'environ 14 indique des queues plus épaisses que la normale, mais modérées. La variance et le moment d'ordre quatre existent, puisque dof dépasse 2 et 4, et la kurtosis en excès théorique d'une t(14,4) vaut 6/(14,4 − 4) ≈ 0,58. Ce résultat concorde avec le fait stylisé des queues épaisses des rendements financiers (séance 1). Il porte sur la distribution de Mkt-RF conditionnelle aux cinq autres facteurs.

**(iii)** La loi de Cauchy (dof = 1) est implausible. Sa log-vraisemblance (−1 511,27) est inférieure de 112,53 points au maximum, et la courbe y chute très fortement. Elle produirait des valeurs extrêmes bien plus fréquentes que celles observées. La loi quasi normale (dof > 60) est elle aussi moins vraisemblable. À dof ≈ 64,5, la log-vraisemblance (−1 402,33) est inférieure de 3,59 points au maximum, et la courbe continue de baisser vers la droite. Un écart de plus de 1,92 point correspond au seuil de 5 % d'un test RV à un degré de liberté, donc les deux extrêmes sont rejetés au profit de dof ≈ 14.

**(iv)** Notre estimation nous préoccupe peu. Avec dof ≈ 14, la variance et le moment d'ordre quatre sont finis, donc le théorème central limite s'applique et l'estimateur robuste de White, qui exige ces moments, reste convergent. Les t et le test J restent ainsi valides asymptotiquement avec 504 mois. La convergence est toutefois plus lente avec des queues épaisses, et quelques mois extrêmes comme octobre 1987 ou avril 2009 pèsent lourd. Les erreurs-types robustes corrigent l'hétéroscédasticité, pas l'épaisseur des queues, et notre dof porte sur Mkt-RF, pas sur les résidus des 17 régressions sectorielles.

# Références

Anthropic. (2026). *Claude* (version Claude Opus 5.5) [Grand modèle de langage]. https://claude.ai

French, K. R. (s. d.). *Data Library*. Tuck School of Business, Dartmouth College. https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

National Bureau of Economic Research. (2023). *US business cycle expansions and contractions*. https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions

Sheppard, K. (s. d.). *linearmodels* (version 7.1) [Logiciel]. https://bashtage.github.io/linearmodels/

van Norden, S. et Sanford, A. (2023). *Méthodes économétriques en finance* [Diapositives des séances 1 à 8, MATH 60231]. HEC Montréal.